In [174]:
import pandas as pd
import json

### Funções auxiliares

In [175]:
def converter_datas(serie: pd.Series) -> pd.Series:
    """Converte datas nos formatos ISO (YYYY-MM-DD) e BR (DD/MM/YYYY) para um único formato"""
    iso = pd.to_datetime(serie, format='%Y-%m-%d', errors='coerce')
    br = pd.to_datetime(serie, format='%d/%m/%Y', errors='coerce')

    resultado = iso.fillna(br)

    nao_parseadas = resultado.isna().sum()
    ## if nao_parseadas > 0:
        ## log...

    return resultado

In [176]:
def converter_moeda_br(serie: pd.Series) -> pd.Series:
    """Converte valores em reais (R$ 1.250,00) para o formato do Pandas (1250.00) e converte para float"""
    return (serie
            .str.replace(r'R\$\s*', '', regex=True)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .str.strip()
            .astype(float))

In [177]:
def converter_percentual(serie: pd.Series) -> pd.Series:
    """Converte percentuais (%) para decimal"""
    return ((serie
 .str.replace('%', '', regex=False)
 .str.replace(',', '.', regex=False)
 .str.strip()
 .astype(float)
 .div(100)))

In [178]:
def limpar_colunas_string(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Remove espaços duplos em colunas que são strings e aplica strip()"""
    for col in colunas:
        df[col] = (df[col]
                   .str.replace(r'\s+', ' ', regex=True)
                   .str.strip())
    
    return df

### Leitura dos arquivos

In [179]:
vendas_jan2025 = pd.read_csv('vendas_jan2025.csv', sep=';', encoding='latin-1')

with open('vendas_fev2025.json', 'r', encoding='utf-8') as f:
    vendas_fev2025 = pd.json_normalize(json.load(f), sep='_')

vendas_mar2025 = pd.read_excel('vendas_mar2025.xlsx', sheet_name='dados')

### Limpeza dos arquivos

In [180]:
vendas_jan2025 = vendas_jan2025.drop_duplicates()

vendas_jan2025['data_venda'] = converter_datas(vendas_jan2025['data_venda'])
vendas_jan2025['preco_unitario'] = converter_moeda_br(vendas_jan2025['preco_unitario'])
vendas_jan2025['desconto_percentual'] = converter_percentual(vendas_jan2025['desconto_percentual'])

colunas_string = vendas_jan2025.select_dtypes('string').columns
vendas_jan2025 = limpar_colunas_string(vendas_jan2025, colunas_string)
vendas_jan2025['vendedor'] = vendas_jan2025['vendedor'].str.title()

vendas_jan2025 = vendas_jan2025.reset_index(drop=True)


In [181]:
vendas_fev2025 = vendas_fev2025.drop_duplicates()

In [182]:
vendas_fev2025.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 0 to 121
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_transacao         116 non-null    int64  
 1   data_venda           116 non-null    str    
 2   id_loja              116 non-null    int64  
 3   id_produto           116 non-null    int64  
 4   quantidade           116 non-null    int64  
 5   preco_unitario       116 non-null    float64
 6   desconto_percentual  112 non-null    str    
 7   vendedor             99 non-null     str    
 8   forma_pagamento      106 non-null    str    
dtypes: float64(1), int64(4), str(4)
memory usage: 9.1 KB


In [183]:
vendas_fev2025['data_venda'] = pd.to_datetime(vendas_fev2025['data_venda'], format='%d-%m-%Y')

In [ ]:
vendas_fev2025

,id_transacao,data_venda,id_loja,id_produto,quantidade,preco_unitario,desconto_percentual,vendedor,forma_pagamento
0,8083,2025-02-28,104,1003,33,19.35,21.7%,João Pereira,PIX
1,8099,2025-02-14,104,1014,49,55.09,18.1%,Ana Silva,Dinheiro
2,8007,2025-02-21,104,1015,37,116.26,5.8%,Ana Silva,NaN
3,8071,2025-02-03,101,1007,29,17.36,17.0%,Maria Lima,PIX
4,8098,2025-02-03,101,1015,57,44.91,13.4%,Carlos Souza,PIX
...,...,...,...,...,...,...,...,...,...
117,8075,2025-02-10,105,1003,51,11.75,3.4%,Maria Lima,Cartão Débito
118,8038,2025-02-11,101,1015,42,40.76,24.6%,João Pereira,Dinheiro
119,8050,2025-02-21,102,1002,5,85.51,8.6%,Ana Silva,NaN
120,8019,2025-02-21,102,1011,28,80.26,NaN,Carlos Souza,NaN


In [185]:
vendas_fev2025.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 0 to 121
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id_transacao         116 non-null    int64         
 1   data_venda           116 non-null    datetime64[us]
 2   id_loja              116 non-null    int64         
 3   id_produto           116 non-null    int64         
 4   quantidade           116 non-null    int64         
 5   preco_unitario       116 non-null    float64       
 6   desconto_percentual  112 non-null    str           
 7   vendedor             99 non-null     str           
 8   forma_pagamento      106 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(4), str(3)
memory usage: 9.1 KB


In [187]:
vendas_fev2025['desconto_percentual'] = converter_percentual(vendas_fev2025['desconto_percentual'])

In [188]:
vendas_fev2025

,id_transacao,data_venda,id_loja,id_produto,quantidade,preco_unitario,desconto_percentual,vendedor,forma_pagamento
0,8083,2025-02-28,104,1003,33,19.35,0.217,João Pereira,PIX
1,8099,2025-02-14,104,1014,49,55.09,0.181,Ana Silva,Dinheiro
2,8007,2025-02-21,104,1015,37,116.26,0.058,Ana Silva,NaN
3,8071,2025-02-03,101,1007,29,17.36,0.170,Maria Lima,PIX
4,8098,2025-02-03,101,1015,57,44.91,0.134,Carlos Souza,PIX
...,...,...,...,...,...,...,...,...,...
117,8075,2025-02-10,105,1003,51,11.75,0.034,Maria Lima,Cartão Débito
118,8038,2025-02-11,101,1015,42,40.76,0.246,João Pereira,Dinheiro
119,8050,2025-02-21,102,1002,5,85.51,0.086,Ana Silva,NaN
120,8019,2025-02-21,102,1011,28,80.26,NaN,Carlos Souza,NaN


In [189]:
vendas_fev2025.info()

<class 'pandas.DataFrame'>
Index: 116 entries, 0 to 121
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id_transacao         116 non-null    int64         
 1   data_venda           116 non-null    datetime64[us]
 2   id_loja              116 non-null    int64         
 3   id_produto           116 non-null    int64         
 4   quantidade           116 non-null    int64         
 5   preco_unitario       116 non-null    float64       
 6   desconto_percentual  112 non-null    float64       
 7   vendedor             99 non-null     str           
 8   forma_pagamento      106 non-null    str           
dtypes: datetime64[us](1), float64(2), int64(4), str(2)
memory usage: 9.1 KB


In [190]:
colunas_string = vendas_fev2025.select_dtypes('str').columns

In [191]:
vendas_fev2025 = limpar_colunas_string(vendas_fev2025, colunas_string)

In [193]:
vendas_fev2025['vendedor'] = vendas_fev2025['vendedor'].str.title()

In [195]:
vendas_fev2025 = vendas_fev2025.reset_index(drop=True)

In [196]:
vendas_fev2025

,id_transacao,data_venda,id_loja,id_produto,quantidade,preco_unitario,desconto_percentual,vendedor,forma_pagamento
0,8083,2025-02-28,104,1003,33,19.35,0.217,João Pereira,PIX
1,8099,2025-02-14,104,1014,49,55.09,0.181,Ana Silva,Dinheiro
2,8007,2025-02-21,104,1015,37,116.26,0.058,Ana Silva,NaN
3,8071,2025-02-03,101,1007,29,17.36,0.170,Maria Lima,PIX
4,8098,2025-02-03,101,1015,57,44.91,0.134,Carlos Souza,PIX
...,...,...,...,...,...,...,...,...,...
111,8075,2025-02-10,105,1003,51,11.75,0.034,Maria Lima,Cartão Débito
112,8038,2025-02-11,101,1015,42,40.76,0.246,João Pereira,Dinheiro
113,8050,2025-02-21,102,1002,5,85.51,0.086,Ana Silva,NaN
114,8019,2025-02-21,102,1011,28,80.26,NaN,Carlos Souza,NaN


In [197]:
vendas_fev2025.info()

<class 'pandas.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id_transacao         116 non-null    int64         
 1   data_venda           116 non-null    datetime64[us]
 2   id_loja              116 non-null    int64         
 3   id_produto           116 non-null    int64         
 4   quantidade           116 non-null    int64         
 5   preco_unitario       116 non-null    float64       
 6   desconto_percentual  112 non-null    float64       
 7   vendedor             99 non-null     str           
 8   forma_pagamento      106 non-null    str           
dtypes: datetime64[us](1), float64(2), int64(4), str(2)
memory usage: 8.3 KB
